# Comparing the results of multiple 
This document checks if every model has been 

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import random

In [ ]:
from os import listdir
from os.path import isfile

Importing the collected data

In [ ]:
from google.colab import files
uploaded = files.upload()
%ls

Saving binance-coin2h.csv to binance-coin2h.csv
Saving bitcoin-cash2h.csv to bitcoin-cash2h.csv
Saving bitcoin2h.csv to bitcoin2h.csv
Saving cardano2h.csv to cardano2h.csv
Saving chainlink2h.csv to chainlink2h.csv
Saving ethereum2h.csv to ethereum2h.csv
Saving litecoin2h.csv to litecoin2h.csv
Saving ripple2h.csv to ripple2h.csv
Saving stellar2h.csv to stellar2h.csv
binance-coin2h.csv  cardano2h.csv    litecoin2h.csv  stellar2h.csv
bitcoin2h.csv       chainlink2h.csv  ripple2h.csv
bitcoin-cash2h.csv  ethereum2h.csv   sample_data/


## Helper Methods

In [ ]:
def plot_time_series(predicted, true, n_training, filename):
  """
  Plot the time series
  """
  plt.figure(figsize=(8,6)) #plotting
  plt.axvline(x=n_training, color="#ffd166", linestyle='-') #size of the training set

  plt.plot(predicted, label='Predicted Price', color="#118ab2") #predicted plot
  plt.plot(true, label='True Price', color="#06d6a0") #actual plot

  plt.title('Time-Series Prediction', fontsize=16)
  plt.xlabel('Time', fontsize=14)
  plt.ylabel('Price', fontsize=14)

  plt.xlim(0)
  plt.legend()
  plt.show()
  #plt.savefig(filename) 

In [ ]:
def classify(predicted_price, true_price):
    

## LSTM Model


### Model Definition

LSTM Class used: https://pytorch.org/docs/master/generated/torch.nn.LSTM.html#torch.nn.LSTM

Some tutorials: https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html



In [ ]:
class LSTMCustom(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, seq_length):
        super(LSTMCustom, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.input_size = input_size #input size
        self.hidden_size = hidden_size #hidden state
        self.seq_length = seq_length #sequence length

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True) #lstm
        #self.fc_1 =  nn.Linear(hidden_size, 128) #fully connected 1
        #self.fc = nn.Linear(128, num_classes) #fully connected last layer

        #self.relu = nn.ReLU()

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self,x):
        h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        output, (hn, cn) = self.lstm(x, (h_0, c_0)) #lstm with input, hidden, and internal state

        out = self.fc(output[:,-1,:])
        return out

### Model Parameters

In [ ]:
num_epochs = 2000 #1000 epochs
learning_rate = 0.01 #0.001 lr

input_size = 1 #number of features
hidden_size = 32 #number of features in hidden state
num_layers = 8 #number of stacked lstm layers

num_classes = 1 #number of output classes 

look_back = 12

In [ ]:
criterion = torch.nn.MSELoss()  # mean-squared error for regression

## Training Each Crypto

In [ ]:
mm = MinMaxScaler()
ss = StandardScaler()

In [ ]:
def train_test_split_tensor(x, y):
  """
  Custom train/test splitting
  TODO: maybe shorten code by using sklearn.preprocessing.train_test_split
  """
  cutoff = round(x.shape[0] * 0.8)

  # split into train and test
  x_train = x[:cutoff, :]
  x_test = x[cutoff:,:]
  y_train = y[:cutoff, :]
  y_test = y[cutoff:, :]

  # convert to tensor
  x_train = Variable(torch.Tensor(x_train))
  x_test = Variable(torch.Tensor(x_test))
  y_train = Variable(torch.Tensor(y_train))
  y_test = Variable(torch.Tensor(y_test)) 
  
  return x_train, x_test, y_train, y_test, cutoff

In [ ]:
def process_data(df):

  # split into x and y
  x = df.iloc[:, :-1]
  x_raw = ss.fit_transform(x)
  x_2d = []

  for index in range(len(x_raw) - look_back):
    x_2d.append(x_raw[index: index + look_back])

  x_2d = np.array(x_2d)
  y = df.iloc[:,-1:]
  
  #print(x_2d[:5])

  # transform the data
  #x_ss = ss.fit_transform(x)
  y_mm = mm.fit_transform(y)
  

  # get train and test split and convert to tensors
  x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y_mm)

  return x_train, x_test, y_train, y_test, cutoff

In [ ]:
df.iloc[:,:-1]

,sentiment_positive_reddit_t-1,sentiment_positive_reddit_t-2,sentiment_positive_reddit_t-3,sentiment_positive_reddit_t-4,sentiment_positive_reddit_t-5,sentiment_positive_reddit_t-6,sentiment_positive_reddit_t-7,sentiment_positive_reddit_t-8,sentiment_positive_reddit_t-9,sentiment_positive_reddit_t-10,sentiment_positive_twitter_t-1,sentiment_positive_twitter_t-2,sentiment_positive_twitter_t-3,sentiment_positive_twitter_t-4,sentiment_positive_twitter_t-5,sentiment_positive_twitter_t-6,sentiment_positive_twitter_t-7,sentiment_positive_twitter_t-8,sentiment_positive_twitter_t-9,sentiment_positive_twitter_t-10,sentiment_negative_reddit_t-1,sentiment_negative_reddit_t-2,sentiment_negative_reddit_t-3,sentiment_negative_reddit_t-4,sentiment_negative_reddit_t-5,sentiment_negative_reddit_t-6,sentiment_negative_reddit_t-7,sentiment_negative_reddit_t-8,sentiment_negative_reddit_t-9,sentiment_negative_reddit_t-10,sentiment_negative_twitter_t-1,sentiment_negative_twitter_t-2,sentiment_negative_twitter_t-3,sentiment_negative_twitter_t-4,sentiment_negative_twitter_t-5,sentiment_negative_twitter_t-6,sentiment_negative_twitter_t-7,sentiment_negative_twitter_t-8,sentiment_negative_twitter_t-9,sentiment_negative_twitter_t-10,social_volume_reddit_t-1,social_volume_reddit_t-2,social_volume_reddit_t-3,social_volume_reddit_t-4,social_volume_reddit_t-5,social_volume_reddit_t-6,social_volume_reddit_t-7,social_volume_reddit_t-8,social_volume_reddit_t-9,social_volume_reddit_t-10,social_volume_twitter_t-1,social_volume_twitter_t-2,social_volume_twitter_t-3,social_volume_twitter_t-4,social_volume_twitter_t-5,social_volume_twitter_t-6,social_volume_twitter_t-7,social_volume_twitter_t-8,social_volume_twitter_t-9,social_volume_twitter_t-10,mean_price_before,std_price_before
0,5.675626,5.675626,5.675626,5.675626,5.675626,5.675626,5.675626,5.675626,5.675626,5.675626,22.790684,22.790684,22.790684,22.790684,22.790684,22.790684,22.790684,22.790684,22.790684,22.790684,7.074674,7.074674,7.074674,7.074674,7.074674,7.074674,7.074674,7.074674,7.074674,7.074674,3.298774,3.298774,3.298774,3.298774,3.298774,3.298774,3.298774,3.298774,3.298774,3.298774,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,7218.541090,0.0
1,2.318428,2.318428,2.318428,2.318428,2.318428,2.318428,2.318428,2.318428,2.318428,2.318428,13.471490,13.471490,13.471490,13.471490,13.471490,13.471490,13.471490,13.471490,13.471490,13.471490,7.273158,7.273158,7.273158,7.273158,7.273158,7.273158,7.273158,7.273158,7.273158,7.273158,0.709921,0.709921,0.709921,0.709921,0.709921,0.709921,0.709921,0.709921,0.709921,0.709921,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,19.0,19.0,19.0,19.0,19.0,19.0,19.0,19.0,19.0,19.0,7215.993093,0.0
2,3.911258,3.911258,3.911258,3.911258,3.911258,3.911258,3.911258,3.911258,3.911258,3.911258,18.236040,18.236040,18.236040,18.236040,18.236040,18.236040,18.236040,18.236040,18.236040,18.236040,4.733463,4.733463,4.733463,4.733463,4.733463,4.733463,4.733463,4.733463,4.733463,4.733463,2.323480,2.323480,2.323480,2.323480,2.323480,2.323480,2.323480,2.323480,2.323480,2.323480,18.0,18.0,18.0,18.0,18.0,18.0,18.0,18.0,18.0,18.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,26.0,7208.649186,0.0
3,1.522572,1.522572,1.522572,1.522572,1.522572,1.522572,1.522572,1.522572,1.522572,1.522572,12.915566,12.915566,12.915566,12.915566,12.915566,12.915566,12.915566,12.915566,12.915566,12.915566,4.715993,4.715993,4.715993,4.715993,4.715993,4.715993,4.715993,4.715993,4.715993,4.715993,4.229568,4.229568,4.229568,4.229568,4.229568,4.229568,4.229568,4.229568,4.229568,4.229568,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,7196.809674,0.0
4,4.050314,4.050314,4.050314,4.050314,4.050314,4.050314,4.050314,4.050314,4.050314,4.050314,9.198932,9.198932,9.198932,9.198932,9.198932,9.198932,9.198932,9.198932,9.198932,9.198932,3.608992,3.608992,3.608992,3.608992,3.608992,3.608992,3.608992,3.608992,3.608992,3.608992,1.58

In [ ]:
def train(df, lstm):
  """
  Train the lstm
  """
  x_train, x_test, y_train, y_test, cutoff = process_data(df)

  for epoch in range(num_epochs):
    outputs = lstm.forward(x_train) #forward pass
    optimizer.zero_grad() #caluclate the gradient, manually setting to 0
  
    # obtain the loss function
    loss = criterion(outputs, y_train)
  
    loss.backward()
    optimizer.step()

    if epoch % 250 == 0:
      print("Epoch: %d, loss: %1.5f" % (epoch, loss.item())) 

  return lstm, cutoff

In [ ]:
# dir_path = "data/"
# file_list = [f for f in listdir(dir_path) if isfile(join(dir_path, f))]
file_list = [f for f in listdir() if isfile(f)]
print(file_list)

['bitcoin2h.csv', 'ripple2h.csv', 'cardano2h.csv', 'bitcoin-cash2h.csv', 'stellar2h.csv', 'binance-coin2h.csv', 'chainlink2h.csv', 'ethereum2h.csv', 'litecoin2h.csv']


In [ ]:
# LOCAL PATH

# dir_path = "datatest/"
# file_list = [f for f in listdir(dir_path) if isfile(join(dir_path, f))]


In [ ]:
# COLAB

dir_path = ""
file_list = [f for f in listdir() if isfile(f)]

In [ ]:
file_endings = ["5m.csv", "hourly.csv", "daily.csv", "2h.csv", "30m.csv"]

file_ending = file_endings[4] # only use 2h

In [ ]:
results = []

# file_list = [name for name in file_list if file_ending in name]:
for filename in file_list:
  crypto_name = filename.replace(file_ending, "")
  
  df = pd.read_csv(dir_path + filename)

  # remove unused columns
  df = df.drop("datetime", axis=1)

  # exclude_cols = [1]
  # df = df.iloc[:, ~df.columns.isin(df.columns[exclude_cols])]

  # define LSTM class
  lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
  optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

  lstm, cutoff = train(df, lstm)

  df_x_ss = ss.transform(df.iloc[:, :1]) #old transformers

  df_x_2d = []
  for index in range(len(df_x_ss) - look_back):
    df_x_2d.append(df_x_ss[index: index + look_back])
  df_y_mm = mm.transform(df.iloc[:, -1:]) #old transformers

  df_x_2d = np.array(df_x_2d)
  df_x_ss = Variable(torch.Tensor(df_x_2d)) #converting to Tensors
  df_y_mm = Variable(torch.Tensor(df_y_mm))


  train_predict = lstm(df_x_ss) #forward pass
  data_predict = train_predict.data.numpy() #numpy conversion
  
  dataY_plot = df_y_mm.data.numpy()

  data_predict = mm.inverse_transform(data_predict) #reverse transformation
  #print(df.iloc[cutoff:].priceDirection.reset_index(drop=True))

  accuracy = classify(data_predict[cutoff:], df.iloc[cutoff:].priceDirection.reset_index(drop=True))
  print(accuracy)
  dataY_plot = mm.inverse_transform(dataY_plot)

  plot_name = f"{crypto_name}.png"
  plot_time_series(data_predict, dataY_plot, cutoff, plot_name)

  #model_name = f"{crypto_name}.pth"
  #torch.save(lstm.state_dict(), model_name)

  results.append([filename, num_epochs, learning_rate, accuracy])
  break

Epoch: 0, loss: 0.03996
Epoch: 250, loss: 0.00413
Epoch: 500, loss: 0.00413
Epoch: 750, loss: 0.00310
Epoch: 1000, loss: 0.00003
Epoch: 1250, loss: 0.00001
Epoch: 1500, loss: 0.00000
Epoch: 1750, loss: 0.00000


AttributeError: ignored

### Results

In [ ]:
mse = {}

y_true = df.loc[look_back:, ["priceUsd"]].values
y_pred = data_predict
mse["total"] = ((y_pred - y_true)**2).mean()

y_true = df.loc[1:cutoff, ["priceUsd"]].values
y_pred = data_predict[:cutoff]
mse["train"] = ((y_pred - y_true)**2).mean()

y_true = df.loc[cutoff+look_back:, ["priceUsd"]].values
y_pred = data_predict[cutoff:]
mse["test"] = ((y_pred - y_true)**2).mean()

mse = pd.Series(mse)

In [ ]:
mse

total    2.001735e+07
train    9.123570e+03
test     9.975039e+07
dtype: float64

In [ ]:
df.columns

Index(['sentiment_positive_reddit_t-1', 'sentiment_positive_reddit_t-2',
       'sentiment_positive_reddit_t-3', 'sentiment_positive_reddit_t-4',
       'sentiment_positive_reddit_t-5', 'sentiment_positive_reddit_t-6',
       'sentiment_positive_reddit_t-7', 'sentiment_positive_reddit_t-8',
       'sentiment_positive_reddit_t-9', 'sentiment_positive_reddit_t-10',
       'sentiment_positive_twitter_t-1', 'sentiment_positive_twitter_t-2',
       'sentiment_positive_twitter_t-3', 'sentiment_positive_twitter_t-4',
       'sentiment_positive_twitter_t-5', 'sentiment_positive_twitter_t-6',
       'sentiment_positive_twitter_t-7', 'sentiment_positive_twitter_t-8',
       'sentiment_positive_twitter_t-9', 'sentiment_positive_twitter_t-10',
       'sentiment_negative_reddit_t-1', 'sentiment_negative_reddit_t-2',
       'sentiment_negative_reddit_t-3', 'sentiment_negative_reddit_t-4',
       'sentiment_negative_reddit_t-5', 'sentiment_negative_reddit_t-6',
       'sentiment_negative_reddit_t-7',

In [ ]:
resultsdf = pd.DataFrame(results)
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)
print(resultsdf)

input size:  1
epochs:  2000
learning rate:  0.01
hidden size:  32
look back:  12
Empty DataFrame
Columns: []
Index: []


In [ ]:
print(results)

[]
